In [1]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
from pathlib import Path
import glob
import os

In [47]:
ARRESTS_FILE = "./data/arrests_2020-2021.csv"
BLOCKS_FILE = "./data/2020_Census_Blocks_nyc.csv"
OUTPUT_DIR = "./data/output"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [49]:
arrests = pd.read_csv(ARRESTS_FILE)
print("Arrests shape:", arrests.shape)
print(arrests.head())

Arrests shape: (295920, 19)
   ARREST_KEY ARREST_DATE  PD_CD                         PD_DESC  KY_CD  \
0   206898146  01/01/2020  511.0  CONTROLLED SUBSTANCE, POSSESSI  235.0   
1   206894723  01/01/2020  109.0        ASSAULT 2,1,UNCLASSIFIED  106.0   
2   206893584  01/01/2020  268.0              CRIMINAL MIS 2 & 3  121.0   
3   206898330  01/01/2020  750.0                RESISTING ARREST  359.0   
4   206893383  01/01/2020  792.0        WEAPONS POSSESSION 1 & 2  118.0   

                        OFNS_DESC    LAW_CODE LAW_CAT_CD ARREST_BORO  \
0                 DANGEROUS DRUGS  PL 2200300          M           K   
1                  FELONY ASSAULT  PL 1200502          F           B   
2  CRIMINAL MISCHIEF & RELATED OF  PL 1450502          F           K   
3  OFFENSES AGAINST PUBLIC ADMINI  PL 2053000          M           Q   
4               DANGEROUS WEAPONS  PL 265031B          F           B   

   ARREST_PRECINCT  JURISDICTION_CODE AGE_GROUP PERP_SEX       PERP_RACE  \
0           

In [51]:
# Convert arrest date to datetime
arrests["ARREST_DATE"] = pd.to_datetime(arrests["ARREST_DATE"], errors="coerce")

# Keep only rows with valid date and coordinates
arrests = arrests.dropna(subset=["ARREST_DATE", "Longitude", "Latitude"]).copy()

In [53]:
# Create year and month columns
arrests["year"] = arrests["ARREST_DATE"].dt.year
arrests["month"] = arrests["ARREST_DATE"].dt.month

In [55]:
# Get the geometry
arrests_gdf = gpd.GeoDataFrame(
    arrests,
    geometry=gpd.points_from_xy(arrests["Longitude"], arrests["Latitude"]),
    crs="EPSG:4326"
)

In [57]:
blocks = pd.read_csv(BLOCKS_FILE)

print("Blocks shape:", blocks.shape)
print(blocks.head())

Blocks shape: (37588, 9)
                                            the_geom  CB2020  BoroCode  \
0  MULTIPOLYGON (((-74.03995040788483 40.70089063...    1000         1   
1  MULTIPOLYGON (((-74.04387761639944 40.69018767...    1001         1   
2  MULTIPOLYGON (((-73.98495042073655 40.71235553...    2000         1   
3  MULTIPOLYGON (((-73.9799619880113 40.713972699...    1000         1   
4  MULTIPOLYGON (((-73.98187417339794 40.71144411...    2000         1   

    BoroName  CT2020    BCTCB2020            GEOID    Shape_Length  \
0  Manhattan     100  10001001000  360610001001000  6,437.85374521   
1  Manhattan     100  10001001001  360610001001001  4,395.19018343   
2  Manhattan     201  10002012000  360610002012000  2,055.29576202   
3  Manhattan     202  10002021000  360610002021000  1,187.88411083   
4  Manhattan     202  10002022000  360610002022000  2,178.77742688   

        Shape_Area  
0  1,202,838.17013  
1   640,166.352288  
2   263,308.402003  
3   57,115.9366413  
4   

In [59]:
# Convert WKT geometry string to shapely geometry
blocks["geometry"] = blocks["the_geom"].apply(wkt.loads)

blocks_gdf = gpd.GeoDataFrame(blocks, geometry="geometry", crs="EPSG:4326")

In [61]:
# Keep only useful columns
blocks_gdf = blocks_gdf[[
    "CB2020", "BoroName", "CT2020", "geometry", 'BCTCB2020'
]].copy()

In [63]:
if arrests_gdf.crs != blocks_gdf.crs:
    arrests_gdf = arrests_gdf.to_crs(blocks_gdf.crs)

# predicate="within" means each arrest point must fall within a block polygon
arrests_joined = gpd.sjoin(
    arrests_gdf,
    blocks_gdf,
    how="left",
    predicate="within"
)

print("Joined shape:", arrests_joined.shape)
print(arrests_joined[["year", "month", "ARREST_KEY", "ARREST_DATE", "CB2020", "BCTCB2020"]].head())

Joined shape: (295920, 27)
   year  month  ARREST_KEY ARREST_DATE  CB2020     BCTCB2020
0  2020      1   206898146  2020-01-01  4001.0  3.043700e+10
1  2020      1   206894723  2020-01-01  3002.0  2.041100e+10
2  2020      1   206893584  2020-01-01  3002.0  3.090800e+10
3  2020      1   206898330  2020-01-01  1001.0  4.012200e+10
4  2020      1   206893383  2020-01-01  3001.0  2.004300e+10


In [65]:
arrests_summary = (
    arrests_joined
    .dropna(subset=["CB2020"])   # drop arrests that did not match a block
    .groupby([ "year", "month", "BCTCB2020", "CB2020", "CT2020"], as_index=False)
    .agg(total_arrests=("ARREST_KEY", "count"))
)

print("Arrests summary shape:", arrests_summary.shape)
print(arrests_summary.head())

Arrests summary shape: (112097, 6)
   year  month     BCTCB2020  CB2020  CT2020  total_arrests
0  2020      1  1.000201e+10  1000.0   201.0              1
1  2020      1  1.000201e+10  1001.0   201.0              1
2  2020      1  1.000202e+10  1001.0   202.0              1
3  2020      1  1.000202e+10  2000.0   202.0              1
4  2020      1  1.000202e+10  4002.0   202.0              3


In [67]:
# Full joined file if you want to inspect individual arrests after matching
arrests_joined.to_csv(os.path.join(OUTPUT_DIR, "arrests_with_census_block.csv"), index=False)

# Final required arrest-level aggregated dataset
arrests_summary.to_csv(os.path.join(OUTPUT_DIR, "arrests_by_census_block_month_year.csv"), index=False)

print("Saved:")
print("- output/arrests_with_census_block.csv")
print("- output/arrests_by_census_block_month_year.csv")

Saved:
- output/arrests_with_census_block.csv
- output/arrests_by_census_block_month_year.csv
